# 01 · Getting started: LLMs vs tabular foundation models

Welcome. In this workshop we build an **agent that knows when it's being asked to predict something, and hands that job to a model built for it** instead of guessing.

This first notebook sets up the one idea everything else rests on:

> An LLM is great at language, planning and code. It is **not** a predictive model.
> A **tabular foundation model (TFM)** like TabPFN *is* — and it works like any sklearn model, through an API.

We'll use one dataset throughout: **telco customer churn**. 7,043 customers of a telecom company — their plan, tenure, charges — and whether they left (`Churn`).

**You need** (see the README): a Prior Labs API key in `TABPFN_TOKEN` and a Claude API key in `ANTHROPIC_API_KEY`, in a `.env` file at the repo root.

In [2]:
# Run from the repo root or from notebooks/ — this makes `agent` importable either way
import os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "agent").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

# Keys: found in your environment or .env → used as-is; otherwise you're prompted (input hidden, nothing saved to disk)
from agent.keys import ensure_keys
ensure_keys("TABPFN_TOKEN", "ANTHROPIC_API_KEY")

TABPFN_TOKEN       loaded from .env
ANTHROPIC_API_KEY  loaded from .env


{'TABPFN_TOKEN': '.env', 'ANTHROPIC_API_KEY': '.env'}

## 1. The data

The dataset ships pre-split into three files, and the split matters for everything that follows:

| File | What it is | Rows |
|---|---|---|
| `telco_churn_history.csv` | Customers with a known outcome. **The only thing a model may learn from.** | 5,634 |
| `telco_churn_current.csv` | Customers we *don't* know the outcome for — the ones we want to score. | 1,409 |
| `telco_churn_test.csv` | The same 1,409 rows *with* the outcome, so we can check our predictions. | 1,409 |

In [3]:
from agent.data import load_history, load_current, load_test, split_xy

history = load_history()
current = load_current()
test = load_test()

print(history.shape, current.shape, test.shape)
history.head()

(5634, 20) (1409, 19) (1409, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Male,0,No,Yes,41,Yes,No,DSL,Yes,Yes,Yes,No,Yes,No,One year,No,Mailed check,70.20,2894.55,No
1,Male,0,No,No,57,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Credit card (automatic),18.80,1094.35,No
2,Female,0,Yes,No,42,Yes,Yes,Fiber optic,No,No,Yes,Yes,No,No,Month-to-month,Yes,Credit card (automatic),85.90,3729.75,No
3,Male,0,Yes,No,5,Yes,Yes,Fiber optic,No,Yes,No,Yes,No,No,Month-to-month,Yes,Electronic check,85.40,401.10,Yes
4,Female,0,Yes,No,67,Yes,Yes,DSL,Yes,Yes,Yes,No,No,No,Two year,No,Mailed check,65.65,4322.85,No


In [ ]:
# What share of customers churned? This is the "base rate" — the number any prediction has to beat.
history["Churn"].value_counts(normalize=True).round(3)

One thing to know about this dataset: `TotalCharges` is stored as **text** in the raw file, because brand-new customers (tenure 0) have a blank there. `load_history()` already coerces it to a number (blanks become `NaN`). Keep this in mind for later — it's the kind of thing the *agent* will have to notice on its own.

In [4]:
import pandas as pd
raw = pd.read_csv(ROOT / "data/telco_churn_history.csv")
print("raw dtype:   ", raw["TotalCharges"].dtype)
print("cleaned dtype:", history["TotalCharges"].dtype, "| blanks:", history["TotalCharges"].isna().sum())

raw dtype:    object
cleaned dtype: float64 | blanks: 9


## 2. Ask an LLM to predict churn

Let's do the thing people actually do: paste some data into an LLM and ask it who will churn.

We'll show Claude 40 labeled customers as examples, then ask it to label 60 *new* customers. We'll keep the sample small so this runs in a reasonable time and doesn't cost much — and that smallness is itself part of the point.

In [5]:
import anthropic, json, time
client = anthropic.Anthropic()
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5-5")

N_EXAMPLES, N_SCORE = 40, 60
examples = history.sample(N_EXAMPLES, random_state=0)
to_score = test.sample(N_SCORE, random_state=1)
X_score, y_true = split_xy(to_score)

prompt = f"""You are a churn analyst. Here are {N_EXAMPLES} customers of a telecom company and whether they churned:

{examples.to_csv(index=False)}

Now, for each of these {N_SCORE} customers, estimate the probability (0 to 1) that they will churn.
Reply with ONLY a JSON list of {N_SCORE} numbers, in order, nothing else.

{X_score.to_csv(index=False)}"""

t0 = time.perf_counter()
resp = client.messages.create(model=MODEL, max_tokens=2000, messages=[{"role": "user", "content": prompt}])
llm_seconds = time.perf_counter() - t0
text = "".join(b.text for b in resp.content if b.type == "text")   # skip thinking blocks, join text blocks
llm_probs = json.loads(text[text.index("["): text.rindex("]") + 1])   # tolerate ```json fences or a stray sentence
print(f"{len(llm_probs)} predictions in {llm_seconds:.1f}s")
llm_probs[:10]

60 predictions in 8.1s


[0.02, 0.35, 0.04, 0.02, 0.3, 0.35, 0.55, 0.15, 0.01, 0.02]

In [6]:
from sklearn.metrics import roc_auc_score

y_bin = (y_true == "Yes").astype(int)
llm_auc = roc_auc_score(y_bin, llm_probs)
print(f"LLM ROC AUC on {N_SCORE} customers: {llm_auc:.3f}   (0.5 = coin flip, 1.0 = perfect)")

LLM ROC AUC on 60 customers: 0.791   (0.5 = coin flip, 1.0 = perfect)


Not terrible — the LLM has seen a lot of churn datasets and knows that month-to-month contracts and fiber optic are risk signals. But notice:

- It only saw **40 examples**. It can't see 5,000 — they won't fit, and even if they did, it wouldn't *fit* them the way a model does.
- The probabilities are **vibes**, not calibrated. A "0.7" from the LLM doesn't mean 70% of such customers churn.
- It took several seconds and a few thousand tokens for 60 rows. Score 1,409 customers this way and you're waiting minutes, and you'll get a different answer each run.

## 3. Ask a tabular foundation model instead

TabPFN is a foundation model *for tables*. It was pre-trained on millions of synthetic datasets to do one thing: given labeled rows, predict labels for new rows — no hyperparameter search, no training loop, in a single forward pass. We call it through the Prior Labs API with the `tabpfn-client` package, which looks exactly like sklearn.

In [7]:
from tabpfn_client import TabPFNClassifier

X_hist, y_hist = split_xy(history)

t0 = time.perf_counter()
tfm = TabPFNClassifier()
tfm.fit(X_hist, y_hist)                       # uploads 5,634 rows; "fit" is instant — there's no training loop
tfm_probs = tfm.predict_proba(X_score)        # scores the same 60 customers
tfm_seconds = time.perf_counter() - t0

yes = list(tfm.classes_).index("Yes")
tfm_auc = roc_auc_score(y_bin, tfm_probs[:, yes])
print(f"TabPFN ROC AUC on the same {N_SCORE} customers: {tfm_auc:.3f}  in {tfm_seconds:.1f}s (fit on all {len(X_hist):,} rows)")

TabPFN ROC AUC on the same 60 customers: 0.802  in 4.2s (fit on all 5,634 rows)


Same 60 customers, side by side:

In [8]:
pd.DataFrame({
    "model": ["Claude (40 examples in the prompt)", "TabPFN (fit on 5,634 rows)"],
    "ROC AUC": [round(llm_auc, 3), round(tfm_auc, 3)],
    "seconds": [round(llm_seconds, 1), round(tfm_seconds, 1)],
    "calibrated probabilities": ["no", "yes"],
    "rows it can learn from": ["whatever fits in the prompt", "up to 1M"],
}).set_index("model")

,ROC AUC,seconds,calibrated probabilities,rows it can learn from
model,,,,
Claude (40 examples in the prompt),0.791,8.1,no,whatever fits in the prompt
"TabPFN (fit on 5,634 rows)",0.802,4.2,yes,up to 1M


## 4. Now score everyone

The fit is cached on the API, so scoring all 1,409 current customers is one call. This is the thing an LLM fundamentally can't do for you.

In [9]:
t0 = time.perf_counter()
p_all = tfm.predict_proba(current)[:, yes]
print(f"scored {len(current):,} customers in {time.perf_counter() - t0:.1f}s")

scored = current.assign(p_churn=p_all.round(3)).sort_values("p_churn", ascending=False)
scored[["tenure", "Contract", "InternetService", "PaymentMethod", "MonthlyCharges", "p_churn"]].head(10)

scored 1,409 customers in 1.2s


,tenure,Contract,InternetService,PaymentMethod,MonthlyCharges,p_churn
307,1,Month-to-month,Fiber optic,Electronic check,94.00,0.939
2,1,Month-to-month,Fiber optic,Electronic check,74.45,0.911
249,1,Month-to-month,Fiber optic,Electronic check,93.30,0.910
96,1,Month-to-month,Fiber optic,Electronic check,89.30,0.910
914,1,Month-to-month,Fiber optic,Electronic check,72.10,0.908
1245,1,Month-to-month,Fiber optic,Electronic check,69.55,0.898
211,1,Month-to-month,Fiber optic,Electronic check,69.85,0.894
1086,1,Month-to-month,Fiber optic,Electronic check,90.55,0.893
572,1,Month-to-month,Fiber optic,Electronic check,78.95,0.884
1104,1,Month-to-month,Fiber optic,Electronic check,80.55,0.884


In [10]:
# And since test.csv has the real outcomes for exactly these rows, in the same order:
full_auc = roc_auc_score((test["Churn"] == "Yes").astype(int), p_all)
print(f"TabPFN ROC AUC on all 1,409 held-out customers: {full_auc:.3f}")

TabPFN ROC AUC on all 1,409 held-out customers: 0.858


## What to take away

| | LLM | Tabular foundation model |
|---|---|---|
| Good at | language, reasoning, planning, writing code, *deciding what to do* | predicting a column from other columns |
| Learns from | what fits in the prompt | thousands to a million labeled rows |
| Output | text (numbers in it are guesses) | calibrated probabilities / values |
| Cost per row | tokens | fractions of a cent, batched |

The lesson of this workshop isn't "don't use LLMs". It's **use each for what it's for**: the LLM is the analyst — it reads the question, cleans the data, decides what's needed — and when the need is a prediction, it hands off to a TFM.

Next: [02 · prediction or lookup?](02_prediction_decision.ipynb) — teaching the LLM to recognise *when* a question is a prediction task at all.